In [ ]:
import random
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from matplotlib.animation import FuncAnimation, PillowWriter
from IPython.display import HTML

In [ ]:
# Buffon's needle estimation for pi
class Buffon:

    def __init__(self, needles=100, needle_length=1, line_gap=1, width=10, height=10):

        # constants
        self.l = needle_length
        self.t = line_gap
        self.x_range = width / 2
        self.y_range = height / 2

        # initialise plot
        plt.style.use("dark_background")
        self.fig, self.ax = plt.subplots(ncols=2, figsize=(10, 5), dpi=200)
        self.fig.suptitle(r"Buffon's Needle Estimation for $\pi$", fontsize=22)
        plt.subplots_adjust(left=0.05, right=0.95, top=0.85, bottom=0.1, hspace=0.2, wspace=0.2)

        # ax[0] needle drop plot
        self.ax[0].set_xlim(-self.x_range, self.x_range)
        self.ax[0].set_ylim(-self.y_range, self.y_range)
        self.ax[0].get_xaxis().set_ticks([])
        self.ax[0].get_yaxis().set_ticks([])

        half_lines = int(self.x_range / self.t)
        # self.lines must always include 0
        self.lines = np.arange(-half_lines, half_lines + 1) * self.t
        for line in self.lines:   
            self.ax[0].axvline(line, alpha=0.5)

        self.needle_title = self.ax[0].set_title("Needles Dropped", fontsize=18, x=1, ha="right")
        self.ax[0].set_xlabel(rf"Needle length $\ell = {self.l}$, line gap $t = {self.t}$", fontsize=12)
        self.ax[0].set_ylabel(' ', fontsize=12)

        # ax[1] pi estimate graph
        self.ax[1].axhline(np.pi, linewidth=1.5)
        self.ax[1].grid(True, which="major", linewidth=1.2, alpha=0.5)

        self.ax_pi = self.ax[1].twinx()
        self.ax_pi.set_ylim(self.ax[1].get_ylim())
        self.ax_pi.set_yticks([np.pi])
        self.ax_pi.set_yticklabels([r"$\pi$"])
        self.ax_pi.tick_params(axis='y', length=0)

        self.graph_title = self.ax[1].set_title(r"$\pi \approx$", fontsize=18, x=0, ha="left")
        self.ax[1].set_xlabel("Number of needles", fontsize=12)
        self.ax[1].set_ylabel(r"$\pi$ estimate", fontsize=12)

        # variables
        self.drops = 0
        self.crosses = 0
        self.pi_list = []
        self.line, = self.ax[1].plot([], [], color="red", linewidth=1.5)
        self.needles_per_frame = needles

    # drop a needle
    def drop(self, drops=1, save=False):            

        for _ in range(drops):
            self.drops += 1
            
            # generate random needle
            x1 = random.uniform(-self.x_range, self.x_range)
            y1 = random.uniform(-self.y_range, self.y_range)
            theta = random.uniform(-np.pi, np.pi)
            x2 = x1 + self.l * np.cos(theta)
            y2 = y1 + self.l * np.sin(theta)
    
            # indicate whether needle crosses line(s)
            crossings = abs(np.floor(x2 / self.t) - np.floor(x1 / self.t))
            if crossings > 0:
                cross_colour = "blue"
                self.crosses += crossings
            else:
                cross_colour = "red"
            
            self.ax[0].plot([x1, x2], [y1, y2], color=cross_colour)

            # add new pi estimate
            if self.crosses == 0:
                self.pi_list.append(float("nan"))
            else:
                self.pi_list.append(2 * self.l * self.drops / (self.t * self.crosses))

        self._update_graph()

        if save:
            self.fig.savefig(f"Buffon's_Needle_{self.drops}_Drops.png", dpi=300, bbox_inches="tight")

    # animte simulation
    def animate(self, needles=10, needles_per_frame=1, save=False, fps=5):

        self.needles_per_frame = needles_per_frame
        self.frames = int(needles / needles_per_frame)

        anim = FuncAnimation(self.fig, self._update, frames=self.frames+1, init_func=self._graph_init, blit=False)

        if save:
            anim.save(f"Buffon's_Needle_{needles}_Drops.gif", writer=PillowWriter(fps=fps))
            plt.close(self.fig)
            return
        else:
            return HTML(anim.to_jshtml())

    # pi esimate graph initialiser
    def _graph_init(self):
        self._update_graph()
        return self.line,

    # update single ax[1] frame
    def _update_graph(self):

        if self.drops == 1:
            self.ax[1].scatter(1, self.pi_list[0], s=1.5, color="red", zorder=10)
            # force ax[1] to include initial dot by adding invisible line
            self.ax[1].plot([1, 1], [self.pi_list[0], self.pi_list[0]], alpha=0)
        else:
            self.line.set_data(range(1, self.drops + 1), self.pi_list)
            
        self.ax[1].relim()
        self.ax[1].autoscale_view()
        self.ax_pi.set_ylim(self.ax[1].get_ylim())

        self.needle_title.set_text(f"{self.drops} Needles Dropped")
        if len(self.pi_list) > 1:
            self.graph_title.set_text(rf"$\pi \approx {self.pi_list[-1]:.5g}$")

    # update ax[1] plot (called within FuncAnimation)
    def _update(self, frame):

        if frame == 0:
            return self.line,

        self.drop(self.needles_per_frame)

        return self.line,

In [ ]:
bn = Buffon(needle_length=2)
bn.animate(needles=1000, needles_per_frame=25, save=True, fps=4)

In [ ]:
bn = Buffon(needle_length=0.2)
bn.drop(5000, save=True)